# Etapa 1 – Upload do SQLite para o Storage via notebook

In [ ]:
from azure.storage.blob import BlobServiceClient

In [ ]:
# Connection String do Blob Storage (vá em Storage > Access Keys > Connection string)
# SUBSTITUIR OS DADOS DA CONTA!!!
# SUBSTITUIR OS CAMINHO DO BANCO DE DADOS!!!
# Dataset disponível em https://www.kaggle.com/datasets/terencicp/e-commerce-dataset-by-olist-as-an-sqlite-database?resource=download

conn_str = "DefaultEndpointsProtocol=https;AccountName=...;AccountKey=...;EndpointSuffix=core.windows.net"
container_name = "landing"
local_file_path = "data/db.sqlite"
blob_path = "meubanco.db"

In [ ]:
# Conecta
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_path)


In [ ]:
# Upload
with open(local_file_path, "rb") as data:
    blob_client.upload_blob(data, overwrite=True)

print("Upload realizado com sucesso!")

# Etapa 2 – Montar ou acessar o storage no Databricks

In [ ]:
# SUBSTITUIR OS DADOS DA CONTA!!!

configs = {
  "fs.azure.account.key.<NOMEDACONTA>.blob.core.windows.net": "<CHAVEDEACESSO>"
}

dbutils.fs.mount(
  source = "wasbs://landing@<NOMEDACONTA>.blob.core.windows.net",
  mount_point = "/mnt/landing",
  extra_configs = configs
)

# Etapa 3 – Ler SQLite e salvar CSVs na landing zone

In [ ]:
import sqlite3
import pandas as pd
import os


In [ ]:
# Caminho local do DBFS (Databricks monta o blob em /dbfs/mnt)
db_path = "/dbfs/mnt/landing/meubanco.db"
output_base = "/dbfs/mnt/landing/csv_output"

In [ ]:
# Conecta ao SQLite
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [ ]:
# Lista as tabelas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

In [ ]:
# Exporta cada tabela como CSV
for (table,) in tables:
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    output_path = f"{output_base}/{table}.csv"
    df.to_csv(output_path, index=False)

conn.close()
print("Tabelas exportadas com sucesso!")